# 04 — Continuous Evaluation and Production Feedback


## Mission

Turn release evaluation into an evidence loop. Production telemetry provides ongoing signals—it does not prove correctness. Trace enough to explain stage behavior while minimizing sensitive content.


In [ ]:
from collections import Counter
import json, os, re, time
from pathlib import Path
import pandas as pd

from evaluation_contracts import *

pd.set_option("display.max_colwidth", 90)
corpus = load_corpus()
golden = load_cases()
print(f"Loaded {len(corpus)} corpus chunks and {len(golden)} golden cases.")


## 1. Privacy-aware trace contract

Choose per-use-case modes: metadata-only by default, sampled and redacted for diagnosis, tightly controlled full-debug only when justified. The schema rejects content in metadata-only events.


In [ ]:
from evaluation_contracts import TraceEvent
from pydantic import ValidationError

try:
    TraceEvent(trace_id="t",query_id="q",stage="retrieval",mode="metadata_only",timestamp_ms=0,duration_ms=1,index_version="lexical-v1",policy_version="policy-2026-08",content_sample="forbidden")
except ValidationError as exc:
    print("Fail-closed privacy validation:", str(exc).splitlines()[0])


## 2. Capture the whole path

Retrieval-only tracing cannot explain failures introduced by reranking, context assembly, generation, validation, or rendering.


In [ ]:
def run_instrumented(query_id, query, *, mode="metadata_only", unusual=False, feedback=None, safety=False, slice_="direct_fact"):
    import hashlib
    stages = ["retrieval","reranking","context","generation","validation","rendering"]
    docs = [c.chunk_id for c in corpus if c.status == "current"][:4]
    events=[]
    for i, stage in enumerate(stages):
        events.append(TraceEvent(trace_id=f"trace-{query_id}", query_id=query_id, stage=stage, mode=mode,
            timestamp_ms=1_725_000_000_000+i*20, duration_ms=12+i*4,
            model="offline-teaching-rag" if stage=="generation" else None,
            prompt_version="answer-v3" if stage=="generation" else None,
            index_version="lexical-v1", policy_version="policy-2026-08",
            chunk_ids=docs if stage in {"retrieval","reranking","context"} else [],
            citations=docs[:1] if stage in {"validation","rendering"} else [],
            decision="abstain" if unusual and stage in {"validation","rendering"} else ("answer" if stage in {"validation","rendering"} else None),
            candidate_count=12 if stage=="retrieval" else None,
            token_count=220 if stage=="generation" else None,
            estimated_cost_usd=0.0018 if stage=="generation" else None,
            status="warning" if unusual and stage=="validation" else "ok",
            content_sample="[REDACTED] sampled diagnostic" if mode=="sampled_redacted" else None))
    return {"query_id":query_id,"query_hash":hashlib.sha256(query.encode()).hexdigest()[:16],"events":[e.model_dump() for e in events],
            "unusual":unusual,"feedback":feedback,"safety":safety,"risk":"high" if safety else "low","slice":slice_,
            "empty_retrieval":False,"abstained":unusual,"citation_failure":safety,"top_source":"people/leave-v2.md"}

trace = run_instrumented("001","How many leave days carry over?", mode="metadata_only")
display(pd.DataFrame(trace["events"])[["stage","duration_ms","candidate_count","token_count","status"]])


## 3. Sampling and review queues

Combine random coverage with targeted sampling: unusual or low-confidence traces, negative feedback, safety events, and high-risk work. Sampling only errors misses silent failures; sampling everything creates privacy and cost risk.


In [ ]:
production = [
 run_instrumented("001","leave carryover"),
 run_instrumented("002","unknown policy", unusual=True, slice_="unanswerable"),
 run_instrumented("003","support SLA", feedback="negative", slice_="near_duplicate"),
 run_instrumented("004","other tenant data", safety=True, slice_="tenant_boundary"),
 run_instrumented("005","TLS requirement", slice_="exact_identifier"),
 run_instrumented("006","discount approval", mode="sampled_redacted", slice_="high_risk_policy"),
]
def should_review(t, random_ids={"005"}):
    reasons=[]
    if t["query_id"] in random_ids: reasons.append("random_sample")
    if t["unusual"]: reasons.append("unusual_or_low_confidence")
    if t["feedback"]=="negative": reasons.append("negative_feedback")
    if t["safety"]: reasons.append("safety_event")
    if t["risk"]=="high": reasons.append("high_risk")
    return reasons
review_queue=[{"query_id":t["query_id"],"reasons":should_review(t)} for t in production if should_review(t)]
display(pd.DataFrame(review_queue))
display(pd.Series([t["slice"] for t in production]).value_counts().rename("production slice count"))


## 4. Online labels can arrive later

Online evaluation is not limited to reference-free judges. Delayed expert review, user outcomes, case resolution, citations, and business/process outcomes can all become labels—subject to bias and privacy review.


In [ ]:
online_signals = pd.DataFrame([
 ("immediate","empty retrieval / citation validity","deterministic"),
 ("minutes","user feedback / escalation","behavioral, selection-biased"),
 ("hours","expert review","human-labelled"),
 ("days","ticket resolution / rollback","delayed outcome"),
 ("weeks","renewal or compliance outcome","business/process label; confounded"),
], columns=["arrival","example","interpretation"])
display(online_signals)


## 5. Promote reviewed failures into regression cases

Promotion is a controlled data operation. The reviewer supplies answerability, evidence, reference, slice, and rationale; the trace never becomes golden automatically.


In [ ]:
def promote_to_regression_case(reviewed_trace):
    """Validate an expert-reviewed trace annotation against the shared case contract."""
    return EvalCase.model_validate(reviewed_trace)

reviewed_failure = promote_to_regression_case({
    "case_id":"prod-2026-08-004", "query":"What is the current leave carryover policy?", "answerable":True,
    "expected_document_ids":["leave-policy-v2","leave-policy-v1"],
    "required_evidence_ids":["leave-policy-v2#carryover"],
    "relevant_evidence_ids":["leave-policy-v2#carryover","leave-policy-v1#carryover"],
    "reference_answer":"At most five days carry over and expire March 31.", "slice":"production_regression",
    "risk":"medium", "corpus_version":"corpus-2026-08", "index_version":"lexical-v1",
    "review_status":"reviewed", "reviewer_rationale":"Expert confirmed a stale-source production failure."})

promoted = golden + [reviewed_failure]
assert validate_dataset(promoted, corpus) == []
print(json.dumps(reviewed_failure.model_dump(), indent=2))
print("Persist to a reviewed pull request—not from the notebook execution path.")


## 6. Drift indicators are alarms, not diagnoses


In [ ]:
daily = pd.DataFrame([
 {"day":"2026-08-28","empty_rate":.02,"abstention_rate":.08,"top_source_share":.25,"mean_candidates":11.8,"citation_fail_rate":.01,"p95_latency_ms":740,"cost_per_query":.003,"high_risk_share":.08},
 {"day":"2026-08-29","empty_rate":.03,"abstention_rate":.09,"top_source_share":.27,"mean_candidates":11.5,"citation_fail_rate":.01,"p95_latency_ms":755,"cost_per_query":.0031,"high_risk_share":.08},
 {"day":"2026-08-30","empty_rate":.11,"abstention_rate":.18,"top_source_share":.51,"mean_candidates":7.2,"citation_fail_rate":.05,"p95_latency_ms":910,"cost_per_query":.0044,"high_risk_share":.15},
])
display(daily)
alerts = {col: float(daily.iloc[-1][col] - daily.iloc[:-1][col].mean()) for col in ["empty_rate","abstention_rate","top_source_share","citation_fail_rate","p95_latency_ms","cost_per_query","high_risk_share"]}
display(pd.Series(alerts, name="latest minus baseline"))


## 7. One release report, explicit decision

Hard blockers and quality warnings must remain distinguishable.


In [ ]:
release_report = {
 "dataset":{"version":"golden-2026-08","cases":len(golden),"slices":len(set(c.slice for c in golden))},
 "retrieval":{"recall_at_5":0.91,"multi_evidence_completeness":0.86},
 "generation":{"faithfulness":0.93,"correctness":0.89},
 "citations":{"validity":1.0,"correctness":0.94,"completeness":0.90},
 "answerability":{"false_answer_rate":0.02,"false_abstention_rate":0.04},
 "safety":{"cross_tenant_leaks":0,"critical_attack_successes":0},
 "operations":{"p95_latency_ms":780,"cost_per_supported_answer_usd":0.014},
 "judge":{"prompt_version":"correctness-v2","kappa":0.80},
 "blockers":[], "warnings":["multi-evidence completeness below 0.90 target"],
}
release_report["decision"] = "BLOCK" if release_report["blockers"] else "REVIEW_WARNINGS"
print(json.dumps(release_report, indent=2))


## 8. Technology landscape

OpenTelemetry-style spans provide portable telemetry primitives. Phoenix, LangSmith, and MLflow connect traces to experiments; Ragas and DeepEval provide evaluation abstractions; Giskard and Garak focus on testing/red-team workflows. Select tools after defining contracts, privacy modes, and gates.


## Exercises and production upgrades

1. Add a source-distribution alert and investigate whether it is data drift or a ranking change.
2. Redact tenant and personal identifiers in sampled trace content.
3. Add delayed ticket-resolution labels and document selection bias.
4. Store release reports and per-case results as immutable CI artifacts.

**Checkpoint:** continuous evaluation supplies ongoing, imperfect evidence. Review converts selected traces into labels; versioned regression tests close the loop.
